# Groundlens × FACTS Grounding

Deterministic geometric grounding check (SGI) evaluated against a configurable ensemble of
frontier LLM judges on the FACTS Grounding public set. Cached, resumable, GPU-ready.

In [ ]:
%pip install -q "groundlens>=2026.7.6" anthropic openai google-generativeai scikit-learn matplotlib seaborn pandas pyarrow scipy

In [ ]:
from __future__ import annotations

import csv
import getpass
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Callable, Sequence

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
@dataclass(frozen=True)
class JudgeModel:
    """A frontier model used either as generator or as grounding judge."""

    provider: str  # one of {"anthropic", "openai", "google"}
    model: str     # provider-specific model identifier
    name: str      # short label used in cache keys, columns and figures


@dataclass(frozen=True)
class Config:
    """Experiment configuration. Edit ``judges`` to use one, two or three models."""

    n_examples: int = 400
    generator: JudgeModel = JudgeModel("anthropic", "claude-sonnet-4-6", "gen")
    judges: tuple[JudgeModel, ...] = (
        JudgeModel("anthropic", "claude-sonnet-5", "claude"),
        # JudgeModel("openai", "gpt-5.1", "gpt"),          # uncomment + set a current id
        # JudgeModel("google", "gemini-3-pro", "gemini"),  # uncomment + set a current id
    )
    include_closed_book: bool = True   # closed-book arm as a source of ungrounded answers
    max_workers: int = 8               # concurrent API calls
    chunk_size: int = 1600             # characters per passage (~400 tokens)
    overlap: int = 200                 # character overlap between passages
    topk: int = 3                      # passages averaged for the top-k SGI variant
    fast_encoder: bool = False         # True -> MiniLM (CPU); False -> arctic-embed-l (GPU)
    seed: int = 20260707
    responses_cache: str = "facts_responses.csv"
    judgments_cache: str = "facts_judgments.csv"


CONFIG: Config = Config()
print(f"generator: {CONFIG.generator.model} | judges: {[j.name for j in CONFIG.judges]}")

In [ ]:
_CLIENTS: dict[str, object] = {}
_ENV: dict[str, str] = {
    "anthropic": "ANTHROPIC_API_KEY",
    "openai": "OPENAI_API_KEY",
    "google": "GOOGLE_API_KEY",
}


def load_key(provider: str) -> str:
    """Resolve an API key from the environment, Colab secrets, or an interactive prompt."""
    env: str = _ENV[provider]
    if os.environ.get(env):
        return os.environ[env]
    try:
        from google.colab import userdata  # type: ignore
        value: str | None = userdata.get(env)
        if value:
            os.environ[env] = value
            return value
    except Exception:
        pass
    os.environ[env] = getpass.getpass(f"{env}: ")
    return os.environ[env]


def _client(provider: str) -> object:
    """Return a cached provider client, creating it (and requesting its key) on first use."""
    if provider in _CLIENTS:
        return _CLIENTS[provider]
    if provider == "anthropic":
        from anthropic import Anthropic
        client: object = Anthropic(api_key=load_key(provider))
    elif provider == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=load_key(provider))
    elif provider == "google":
        import google.generativeai as genai
        genai.configure(api_key=load_key(provider))
        client = genai
    else:
        raise ValueError(f"unknown provider: {provider}")
    _CLIENTS[provider] = client
    return client


def call_model(model: JudgeModel, system: str, user: str, max_tokens: int = 512, retries: int = 5) -> str:
    """Return the text completion for a system+user prompt, retrying transient failures."""
    for attempt in range(retries):
        try:
            if model.provider == "anthropic":
                resp = _client("anthropic").messages.create(
                    model=model.model, max_tokens=max_tokens, system=system,
                    messages=[{"role": "user", "content": user}],
                )
                return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text").strip()
            if model.provider == "openai":
                resp = _client("openai").chat.completions.create(
                    model=model.model, max_tokens=max_tokens,
                    messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
                )
                return (resp.choices[0].message.content or "").strip()
            if model.provider == "google":
                gm = _client("google").GenerativeModel(model.model, system_instruction=system)
                return (gm.generate_content(user).text or "").strip()
            raise ValueError(f"unknown provider: {model.provider}")
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 * (attempt + 1))
    return ""

In [ ]:
def run_cached(
    tasks: Sequence[object],
    key_of: Callable[[object], str],
    work: Callable[[object], dict | None],
    cache_csv: str,
    columns: Sequence[str],
    workers: int = CONFIG.max_workers,
    label: str = "work",
) -> pd.DataFrame:
    """Run ``work`` over ``tasks`` in parallel, appending each result to a CSV (crash-safe, resumable)."""
    done: dict[str, dict] = {}
    if os.path.exists(cache_csv):
        for _, row in pd.read_csv(cache_csv).iterrows():
            done[str(row["key"])] = row.to_dict()
    todo: list[object] = [t for t in tasks if key_of(t) not in done]
    print(f"{label}: {len(done)} cached, {len(todo)} to do")
    lock: threading.Lock = threading.Lock()
    handle = open(cache_csv, "a", newline="")
    writer = csv.DictWriter(handle, fieldnames=list(columns))
    if os.path.getsize(cache_csv) == 0:
        writer.writeheader(); handle.flush()
    results: dict[str, dict] = dict(done)

    def _run(task: object) -> dict | None:
        try:
            row = work(task)
        except Exception:
            return None
        if not row:
            return None
        row["key"] = key_of(task)
        return row

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_run, t) for t in todo]
        for i, fut in enumerate(as_completed(futures), start=1):
            row = fut.result()
            if row is None:
                continue
            with lock:
                writer.writerow({c: row.get(c) for c in columns}); handle.flush()
            results[row["key"]] = row
            if i % 50 == 0:
                print(f"  {label}: {i}/{len(todo)}")
    handle.close()
    return pd.DataFrame(list(results.values()))

## Data — FACTS Grounding public examples

In [ ]:
_PARQUET: str = (
    "https://huggingface.co/datasets/google/FACTS-grounding-public/"
    "resolve/refs%2Fconvert%2Fparquet/examples/public/0000.parquet"
)


def load_examples(n: int, seed: int) -> pd.DataFrame:
    """Load and sample the public FACTS Grounding examples (system instruction, request, context)."""
    try:
        frame: pd.DataFrame = pd.read_parquet(_PARQUET)
    except Exception:
        from datasets import load_dataset
        frame = load_dataset("google/FACTS-grounding-public", "examples", split="public").to_pandas()
    frame = frame.dropna(subset=["user_request", "context_document"]).reset_index(drop=True)
    return frame.sample(min(n, len(frame)), random_state=seed).reset_index(drop=True)


examples: pd.DataFrame = load_examples(CONFIG.n_examples, CONFIG.seed)
SYS: list[str] = examples["system_instruction"].tolist()
REQ: list[str] = examples["user_request"].tolist()
CTX: list[str] = examples["context_document"].tolist()
print(f"{len(examples)} examples | context chars: "
      f"median {int(examples.context_document.str.len().median())}, "
      f"max {int(examples.context_document.str.len().max())}")

## Generation — answers from the generator model (grounded and closed-book arms)

In [ ]:
_CLIP: int = 48000  # cap prompt characters on very long documents


def _clip(text: str, limit: int = _CLIP) -> str:
    """Truncate ``text`` to ``limit`` characters."""
    return text[:limit]


def generate(index: int, arm: str) -> dict | None:
    """Produce one answer: grounded (with context) or closed-book (parametric, no context)."""
    if arm == "grounded":
        system: str = (SYS[index] or "") + "\nAnswer using only the context document above."
        prompt: str = f"{REQ[index]}\n\n=== CONTEXT DOCUMENT ===\n{_clip(CTX[index])}"
    else:
        system = "Answer from your own general knowledge. Be specific and concise."
        prompt = REQ[index]
    answer: str = call_model(CONFIG.generator, system, prompt, max_tokens=800)
    return {"i": index, "arm": arm, "response": answer} if answer else None


ARMS: list[str] = ["grounded"] + (["closed_book"] if CONFIG.include_closed_book else [])
gen_tasks: list[tuple[int, str]] = [(i, a) for i in range(len(examples)) for a in ARMS]
responses: pd.DataFrame = run_cached(
    gen_tasks, lambda t: f"{t[0]}:{t[1]}", lambda t: generate(*t),
    CONFIG.responses_cache, ["key", "i", "arm", "response"], label="generate",
)
print(f"responses: {len(responses)}")

## Judging — each configured model labels eligibility and grounding

In [ ]:
JUDGE_SYSTEM: str = (
    "You are a strict evaluator for a grounding benchmark. Given a CONTEXT document, a USER "
    "REQUEST and a RESPONSE, return JSON only: "
    '{"eligible": true|false, "grounded": true|false}. '
    "eligible = the response genuinely attempts the request (false for refusals or off-topic). "
    "grounded = every factual claim is supported by the context or needs no support."
)


def parse_verdict(text: str) -> tuple[bool, bool] | None:
    """Extract the ``eligible`` and ``grounded`` booleans, tolerant to extra text or truncation."""
    low: str = text.lower()

    def flag(field_name: str) -> bool | None:
        match = re.search(rf'"{field_name}"\s*:\s*(true|false)', low)
        return (match.group(1) == "true") if match else None

    eligible, grounded = flag("eligible"), flag("grounded")
    if eligible is None or grounded is None:
        block = re.search(r"\{.*\}", text, re.S)
        if block:
            try:
                data = json.loads(block.group(0))
                eligible, grounded = bool(data.get("eligible")), bool(data.get("grounded"))
            except Exception:
                pass
    return (eligible, grounded) if eligible is not None and grounded is not None else None


_RESP: dict[tuple[int, str], str] = {(int(r.i), r.arm): r.response for r in responses.itertuples()}


def judge(index: int, arm: str, model: JudgeModel) -> dict | None:
    """Score one response with one judge model."""
    prompt: str = (
        f"=== CONTEXT ===\n{_clip(CTX[index])}\n\n=== USER REQUEST ===\n{REQ[index]}"
        f"\n\n=== RESPONSE ===\n{_RESP[(index, arm)]}"
    )
    verdict = parse_verdict(call_model(model, JUDGE_SYSTEM, prompt, max_tokens=256))
    if verdict is None:
        return None
    return {"i": index, "arm": arm, "judge": model.name,
            "eligible": int(verdict[0]), "grounded": int(verdict[1])}


judge_tasks: list[tuple[int, str, JudgeModel]] = [
    (int(r.i), r.arm, m) for r in responses.itertuples() for m in CONFIG.judges
]
verdicts: pd.DataFrame = run_cached(
    judge_tasks, lambda t: f"{t[0]}:{t[1]}:{t[2].name}", lambda t: judge(*t),
    CONFIG.judgments_cache, ["key", "i", "arm", "judge", "eligible", "grounded"], label="judge",
)
verdicts = verdicts[verdicts.judge.isin([m.name for m in CONFIG.judges])]
print(f"per-judge verdicts: {len(verdicts)}")

In [ ]:
def aggregate(per_judge: pd.DataFrame) -> pd.DataFrame:
    """Combine per-judge verdicts into one label per response (FACTS-style majority / any-eligible)."""
    grouped = per_judge.groupby(["i", "arm"])
    out: pd.DataFrame = grouped.agg(
        n_judges=("grounded", "size"),
        eligible=("eligible", "max"),          # disqualified only if every judge says ineligible
        grounded_mean=("grounded", "mean"),
    ).reset_index()
    out["grounded"] = (out["grounded_mean"] > 0.5).astype(int)   # ties (2/2) resolve to ungrounded
    return out


labels: pd.DataFrame = aggregate(verdicts)
data: pd.DataFrame = responses.merge(labels, on=["i", "arm"])
data["request"] = data.i.map(lambda i: REQ[i])
data["context"] = data.i.map(lambda i: CTX[i])
eligible_counts = data[data.eligible == 1]["grounded"].value_counts()
print(f"labelled {len(data)} | eligible {int(data.eligible.sum())} | "
      f"grounded {int(eligible_counts.get(1, 0))} ungrounded {int(eligible_counts.get(0, 0))}")

## Deterministic score — Semantic Grounding Index (SGI), chunked over passages

In [ ]:
import groundlens
from groundlens import DEFAULT_MODEL, LIGHTWEIGHT_MINILM, check, evaluate_batch
from groundlens._internal.thresholds import SGI_REVIEW, SGI_STRONG_PASS

ENCODER: str = LIGHTWEIGHT_MINILM if CONFIG.fast_encoder else DEFAULT_MODEL
print(f"groundlens {groundlens.__version__} | encoder {ENCODER}")


def chunk_text(text: str, size: int = CONFIG.chunk_size, overlap: int = CONFIG.overlap) -> list[str]:
    """Split a document into overlapping character windows (PDF-style passages)."""
    text = text.strip()
    if len(text) <= size:
        return [text]
    windows: list[str] = []
    start: int = 0
    while start < len(text):
        windows.append(text[start:start + size])
        start += size - overlap
    return windows


def score_sgi(frame: pd.DataFrame) -> pd.DataFrame:
    """Attach whole-document and chunked SGI (max, top-k) for every row, via the library."""
    rows: pd.DataFrame = frame.reset_index(drop=True)
    whole_items: list[dict] = [
        {"question": r.request, "response": r.response, "context": r.context} for r in rows.itertuples()
    ]
    rows["sgi_whole"] = [s.value for s in evaluate_batch(whole_items, model=ENCODER)]

    passages: list[dict] = []
    owner: list[int] = []
    for k, r in enumerate(rows.itertuples()):
        for passage in chunk_text(r.context):
            passages.append({"question": r.request, "response": r.response, "context": passage})
            owner.append(k)
    values: np.ndarray = np.array([s.value for s in evaluate_batch(passages, model=ENCODER)])
    owner_arr: np.ndarray = np.array(owner)
    rows["sgi_chunk_max"] = [values[owner_arr == k].max() for k in range(len(rows))]
    rows["sgi_chunk_topk"] = [
        np.sort(values[owner_arr == k])[::-1][: CONFIG.topk].mean() for k in range(len(rows))
    ]
    return rows


scored: pd.DataFrame = score_sgi(data)
print(f"scored {len(scored)} responses")

## Results — SGI vs the judge ensemble (eligible responses)

In [ ]:
def bootstrap_ci(y: np.ndarray, s: np.ndarray, n: int = 3000, seed: int = CONFIG.seed) -> tuple[float, float]:
    """Return the 2.5/97.5 percentile bootstrap interval of the AUROC."""
    rng: np.random.Generator = np.random.default_rng(seed)
    scores: list[float] = []
    for _ in range(n):
        idx: np.ndarray = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) == 2:
            scores.append(roc_auc_score(y[idx], s[idx]))
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return float(lo), float(hi)


eligible: pd.DataFrame = scored[scored.eligible == 1].copy()
y_true: np.ndarray = eligible["grounded"].to_numpy()
print(f"judges: {[m.name for m in CONFIG.judges]} | eligible n={len(eligible)} "
      f"(grounded {int(y_true.sum())}, ungrounded {int((1 - y_true).sum())})\n")
for column, title in [("sgi_whole", "whole-document"),
                      ("sgi_chunk_max", "chunked max (PDF)"),
                      ("sgi_chunk_topk", f"chunked top-{CONFIG.topk}")]:
    auc: float = roc_auc_score(y_true, eligible[column])
    lo, hi = bootstrap_ci(y_true, eligible[column].to_numpy())
    print(f"  AUROC  {title:20s}: {auc:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

In [ ]:
def check_label(value: float) -> str:
    """Map a raw SGI value to the shipped CHECK label."""
    if value >= SGI_STRONG_PASS:
        return "Supported"
    if value >= SGI_REVIEW:
        return "Partly"
    return "Not supported"


eligible["check"] = eligible.sgi_chunk_max.map(check_label)
eligible["judge_label"] = eligible.grounded.map({1: "grounded", 0: "ungrounded"})
print(pd.crosstab(eligible["judge_label"], eligible["check"]), "\n")

auto: pd.DataFrame = eligible[eligible.check != "Partly"]
agree: float = (
    ((auto.check == "Supported") & (auto.judge_label == "grounded"))
    | ((auto.check == "Not supported") & (auto.judge_label == "ungrounded"))
).mean()
print(f"auto-decided by CHECK: {len(auto)}/{len(eligible)} = {len(auto)/len(eligible):.0%}, "
      f"agreeing with the ensemble {agree:.0%}")

## Judge-ensemble agreement — consensus across the configured models

In [ ]:
if len(CONFIG.judges) > 1:
    wide: pd.DataFrame = verdicts.pivot_table(index=["i", "arm"], columns="judge", values="grounded")
    complete: pd.DataFrame = wide.dropna()
    unanimity: float = complete.apply(lambda r: r.nunique() == 1, axis=1).mean()
    print(f"responses judged by all {len(CONFIG.judges)} judges: {len(complete)}")
    print(f"unanimous grounding verdict: {unanimity:.1%}")
    print(complete.mean().rename("share grounded").to_frame())
else:
    print("single judge configured — add models to CONFIG.judges for ensemble agreement stats")

## Figure — SGI distribution by ensemble label

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
_TEAL, _AMBER, _DARK = "#2a9d8f", "#e07a3c", "#2b2b2b"
frame_fig: pd.DataFrame = eligible.assign(label=eligible.judge_label)
auc_fig: float = roc_auc_score(y_true, eligible.sgi_chunk_max)

fig, ax = plt.subplots(figsize=(11, 5.6))
ax.axvspan(0, SGI_REVIEW, color=_AMBER, alpha=0.05)
ax.axvspan(SGI_REVIEW, SGI_STRONG_PASS, color="#9aa0a6", alpha=0.09)
ax.axvspan(SGI_STRONG_PASS, 3.1, color=_TEAL, alpha=0.06)
sns.histplot(frame_fig, x="sgi_chunk_max", hue="label", bins=42,
             palette={"grounded": _TEAL, "ungrounded": _AMBER}, alpha=0.75, ax=ax)
for threshold in (SGI_REVIEW, SGI_STRONG_PASS):
    ax.axvline(threshold, color=_DARK, ls="--", lw=1)
ax.set_xlim(0.3, 3.0)
ax.set_xlabel("SGI (Semantic Grounding Index)")
ax.set_title(f"SGI vs judge ensemble  ·  AUROC {auc_fig:.3f}  ·  n={len(eligible)}",
             loc="left", fontweight="bold", color=_DARK)
plt.tight_layout()
plt.show()